# Dataset Review — Twitter & Amazon

Antes de entrenar cualquier modelo conviene auditar los datos. Este notebook responde tres preguntas por dataset:

1. **¿Están balanceadas las clases?** — Un desbalance severo sesga el entrenamiento.
2. **¿Qué señal textual hay?** — Vocabulario, longitud y bigramas revelan la dificultad del problema.
3. **¿Qué ruido hay que limpiar?** — Duplicados, textos vacíos y tokens irrelevantes.

> **Cómo ejecutar:** `Kernel › Restart & Run All`

In [ ]:
from collections import Counter
from string import punctuation
import re, json, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.style.use('seaborn-v0_8-whitegrid')
NEG, POS, BASE, MID = '#E8505B', '#4ECDC4', '#6C9BCF', '#F0A500'

STOPS = {
    'the','a','an','is','it','in','of','to','and','for','i','my','on',
    'at','with','this','that','be','are','was','have','has','me','we',
    'he','she','they','but','or','not','so','do','if','by','as','from',
    'its','you','your','im','rt','amp','just','get','got','been','will',
    'had','did','all','can','one','out','up','about','more','her','his',
    'their','our','who','what','when','how','there','no','now','then',
}

def tokenize(text):
    tokens = []
    for w in text.split():
        tok = w.strip(punctuation).lower()
        if len(tok) > 1 and tok not in STOPS:
            tokens.append(tok)
    return tokens

def top_n(texts, n=15):
    return Counter(w for t in texts for w in tokenize(t)).most_common(n)

def top_bigrams(texts, n=12):
    bg = []
    for t in texts:
        toks = tokenize(t)
        bg.extend(f'{a} {b}' for a, b in zip(toks, toks[1:]))
    return Counter(bg).most_common(n)

def hbar(ax, items, color, title):
    words, vals = zip(*items)
    ax.barh(list(reversed(words)), list(reversed(vals)), color=color)
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis='y', labelsize=8)
    ax.set_xlabel('frecuencia', fontsize=9)

vader = SentimentIntensityAnalyzer()

---
## 1. Twitter — `bdstar/twitter-sentiment-analysis`

Dataset de tweets en inglés etiquetados como **positive** o **negative**. Es el corpus principal para las prácticas de clasificación de sentimientos de texto corto.

**Características del dominio:**
- Texto muy corto (≤ 280 caracteres)
- Ruido específico: menciones, hashtags, URLs, abreviaturas
- Lenguaje informal, ironía, emojis

In [ ]:
from datasets import load_dataset

ds = load_dataset('bdstar/twitter-sentiment-analysis')
tr, te = ds['train'], ds['test']

def class_counts(split):
    c = Counter(split['label'])
    return {lbl: c.get(lbl, 0) for lbl in ['positive', 'negative', 'neutral']}

tr_c, te_c = class_counts(tr), class_counts(te)

stats = pd.DataFrame({
    'split':    ['train', 'test'],
    'total':    [len(tr), len(te)],
    'positive': [tr_c['positive'], te_c['positive']],
    'negative': [tr_c['negative'], te_c['negative']],
    'neutral':  [tr_c['neutral'],  te_c['neutral']],
})
stats['% pos'] = (stats['positive'] / stats['total'] * 100).round(1)
stats

### 1.1 Distribución de clases

Un **desbalance severo** entre clases hace que el modelo aprenda a predecir siempre la mayoritaria. Con 90 % de ejemplos positivos, un clasificador trivial alcanza 90 % de accuracy sin aprender nada útil.

> **Qué buscamos:** ratio cercano a 1:1, o documentar el desbalance para compensarlo con `class_weight='balanced'` en sklearn o `pos_weight` en PyTorch.

In [ ]:
LABEL_COLORS = {'negative': NEG, 'neutral': MID, 'positive': POS}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, split, name in zip(axes, [tr, te], ['Train', 'Test']):
    counts = Counter(split['label'])
    labels = [l for l in ['negative', 'neutral', 'positive'] if l in counts]
    vals   = [counts[l] for l in labels]
    colors = [LABEL_COLORS[l] for l in labels]
    bars = ax.bar(labels, vals, color=colors)
    ax.bar_label(bars, fmt='{:,.0f}', padding=4)
    ax.set_title(f'Twitter {name} — distribución de clases')
    ax.set_ylabel('ejemplos')
    ax.set_ylim(0, max(vals) * 1.18)
plt.tight_layout()
plt.show()

### 1.2 Longitud de los textos

La longitud media determina el `max_length` óptimo al tokenizar. Truncar muy corto pierde información; truncar muy largo infla el coste computacional.

- **Histograma:** distribución general del corpus
- **Boxplot por clase:** ¿diferente longitud implica diferente sentimiento?
- **p95:** buen candidato para `max_length` — cubre el 95 % de los textos sin overhead innecesario

In [ ]:
len_by_class = {'negative': [], 'positive': []}
for r in tr:
    if r['label'] in len_by_class:
        len_by_class[r['label']].append(len(r['text'].split()))
all_len = len_by_class['negative'] + len_by_class['positive']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

mu  = np.mean(all_len)
p95 = np.percentile(all_len, 95)
axes[0].hist(all_len, bins=40, color=BASE, edgecolor='white')
axes[0].axvline(mu,  color='crimson',    linestyle='--', label=f'media={mu:.1f}')
axes[0].axvline(p95, color='darkorange', linestyle=':',  label=f'p95={p95:.0f}')
axes[0].set_title('Twitter Train — distribución de longitud (pos + neg)')
axes[0].set_xlabel('palabras')
axes[0].set_ylabel('tweets')
axes[0].legend()

bp = axes[1].boxplot(
    [len_by_class['negative'], len_by_class['positive']],
    tick_labels=['negative', 'positive'],
    patch_artist=True,
    medianprops=dict(color='crimson', linewidth=2)
)
for patch, color in zip(bp['boxes'], [NEG, POS]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_title('Twitter — longitud por clase')
axes[1].set_ylabel('palabras')

plt.tight_layout()
plt.show()
print(f'media={mu:.1f}  mediana={int(np.median(all_len))}  p95={p95:.0f}  max={max(all_len)}')

### 1.3 Tokens especiales (ruido de Twitter)

Twitter introduce tokens que **no aportan semántica generalizable** pero consumen espacio en el vocabulario y pueden introducir sesgos (mencionar a un usuario famoso puede correlacionar con negatividad sin ser señal real).

| Token | Impacto | Tratamiento habitual |
|-------|---------|----------------------|
| `@mención` | Sesgo por usuario | → `<USER>` |
| `#hashtag` | Ruido de vocabulario | Quitar `#` o → `<HASHTAG>` |
| URL | Sin semántica | → `<URL>` |

> El preprocesamiento en `preprocessing.py` ya aplica parte de esta normalización.

In [ ]:
tw_texts    = tr['text']
has_mention = [bool(re.search(r'@\w+', t))              for t in tw_texts]
has_hashtag = [bool(re.search(r'#\w+', t))              for t in tw_texts]
has_url     = [bool(re.search(r'https?://\S+|www\.\S+', t)) for t in tw_texts]

n = len(tw_texts)
token_pct = {
    '@menciones': sum(has_mention) / n * 100,
    '#hashtags':  sum(has_hashtag) / n * 100,
    'URLs':       sum(has_url)     / n * 100,
}

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(list(token_pct.keys()), list(token_pct.values()), color=BASE)
ax.bar_label(bars, fmt='{:.1f}%', padding=4)
ax.set_xlim(0, 105)
ax.set_title('Twitter Train — % de tweets con tokens especiales')
ax.set_xlabel('% del total')
plt.tight_layout()
plt.show()

### 1.4 Palabras más frecuentes

Los **unigramas** muestran vocabulario discriminativo: palabras que aparecen mucho en una clase pero poco en la otra son señales directas para el clasificador.

**Limitación:** pierden el contexto. *not happy* contiene `happy`, igual que *very happy*. Los bigramas (siguiente gráfico) capturan esta diferencia.

In [ ]:
pos_tr = [r['text'] for r in tr if r['label'] == 'positive']
neg_tr = [r['text'] for r in tr if r['label'] == 'negative']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hbar(axes[0], top_n(neg_tr), NEG, 'Twitter Train — top palabras (Negative)')
hbar(axes[1], top_n(pos_tr), POS, 'Twitter Train — top palabras (Positive)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hbar(axes[0], top_bigrams(neg_tr), NEG, 'Twitter Train — top bigramas (Negative)')
hbar(axes[1], top_bigrams(pos_tr), POS, 'Twitter Train — top bigramas (Positive)')
plt.tight_layout()
plt.show()

### 1.5 Separabilidad léxica con VADER

VADER asigna un score compuesto en [−1, +1] usando un léxico con reglas de contexto (negaciones, mayúsculas, signos de puntuación). Visualizar su distribución por clase revela:

- **¿Qué tan separables son las clases?** Si las distribuciones se solapan mucho, un léxico solo no alcanzará alta accuracy.
- **¿Dónde falla VADER?** Textos negativos con score ≈ 0 son los casos difíciles: ironía, jerga, ausencia de palabras de sentimiento explícitas.

Los umbrales punteados (±0.05) son los que usa VADER por defecto para clasificar positivo/negativo/neutro.

In [ ]:
vader_scores = {'negative': [], 'positive': []}
for r in tr:
    if r['label'] in vader_scores:
        score = vader.polarity_scores(r['text'])['compound']
        vader_scores[r['label']].append(score)

fig, ax = plt.subplots(figsize=(10, 4))
for lbl, color in [('negative', NEG), ('positive', POS)]:
    ax.hist(vader_scores[lbl], bins=60, alpha=0.65, color=color, label=lbl, density=True)
ax.axvline( 0.05, color='black', linestyle='--', linewidth=1.5, label='umbral +0.05')
ax.axvline(-0.05, color='black', linestyle=':',  linewidth=1.5, label='umbral -0.05')
ax.set_title('Twitter — distribución de scores VADER por clase')
ax.set_xlabel('compound score  (-1 = muy negativo  ·  +1 = muy positivo)')
ax.set_ylabel('densidad')
ax.legend()
plt.tight_layout()
plt.show()

for lbl in ['negative', 'positive']:
    scores = vader_scores[lbl]
    print(f'{lbl:10s}  media={np.mean(scores):+.3f}  std={np.std(scores):.3f}')

### 1.6 Calidad del dataset

Antes de entrenar hay que cuantificar el ruido estructural:

- **Duplicados:** el modelo puede memorizar ejemplos repetidos, inflando el accuracy de validación artificialmente.
- **Textos vacíos:** no aportan señal y pueden generar errores en el pipeline.
- **Textos muy cortos:** con < 3 palabras hay poca señal útil.

In [ ]:
texts    = tr['text']
dupes_tw = len(texts) - len(set(texts))
empty_tw = sum(1 for t in texts if not t.strip())
short_tw = sum(1 for t in texts if len(t.split()) < 3)

print('── Calidad Twitter Train ────────────────────────────')
print(f'  Total                : {len(texts):,}')
print(f'  Duplicados           : {dupes_tw:,}  ({dupes_tw/len(texts)*100:.2f}%)')
print(f'  Vacíos               : {empty_tw:,}')
print(f'  Muy cortos (<3 pal.) : {short_tw:,}')
print()

pd.DataFrame(tr[:8])[['text', 'label']]

---
## 2. Amazon — `McAuley-Lab/Amazon-Reviews-2023` (corpus local)

Reseñas de productos con rating de 1 a 5 estrellas. A diferencia de Twitter:

| | Twitter | Amazon |
|--|---------|--------|
| Longitud | Muy corta (≤ 280 chars) | Párrafos completos |
| Clases | 2 (pos / neg) | 5 ratings |
| Ruido | Menciones, hashtags, URLs | HTML residual, emojis |
| Lenguaje | Informal, ironía | Más formal, descriptivo |

**Estrategia de etiquetado usada en el curso:**
- `rating ≤ 2` → negativo (0)
- `rating ≥ 4` → positivo (1)
- `rating = 3` → descartado (ambiguo)

In [ ]:
corpus_path = pathlib.Path('..') / 'amazon_reviews_corpus.json'
if not corpus_path.exists():
    corpus_path = pathlib.Path('amazon_reviews_corpus.json')

reviews = json.loads(corpus_path.read_text())
df = pd.DataFrame(reviews)
df['rating'] = df['rating'].astype(int)

print(f'Reviews: {len(df):,}  |  Columnas: {list(df.columns)}')
print(f'Ratings disponibles: {sorted(df["rating"].unique().tolist())}')
df.head(4)

### 2.1 Distribución de ratings

Amazon tiene un **sesgo positivo conocido** (distribución en forma de J): la mayoría son 5★ porque los compradores satisfechos tienen más motivación para reseñar. Esto complica el balanceo al binarizar.

> **Implicación:** al construir el split binario hay que submuestrear positivos o usar `class_weight` para compensar la diferencia.

In [ ]:
counts_r = df['rating'].value_counts().sort_index()
colors_r = [NEG if r <= 2 else (MID if r == 3 else POS) for r in counts_r.index]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar([f'{r} estrellas' for r in counts_r.index], counts_r.values, color=colors_r)
ax.bar_label(bars, fmt='{:,.0f}', padding=4)
ax.set_title('Amazon — distribución de ratings')
ax.set_xlabel('rating')
ax.set_ylabel('reviews')
ax.set_ylim(0, counts_r.max() * 1.15)
plt.tight_layout()
plt.show()

print(f'Negativas (<=2): {sum(counts_r[r] for r in counts_r.index if r <= 2):,}')
print(f'Neutras   (=3):  {counts_r.get(3, 0):,}')
print(f'Positivas (>=4): {sum(counts_r[r] for r in counts_r.index if r >= 4):,}')

### 2.2 Longitud de reviews por rating

Las reseñas negativas suelen ser **más largas**: quien está insatisfecho tiene más que explicar. Esta correlación longitud↔sentimiento es un artefacto del dominio, no semántica real.

> **Riesgo:** si no se controla, el modelo puede aprender a clasificar por longitud en lugar de por contenido. Truncar a un `max_length` uniforme mitiga este problema.

In [ ]:
df['n_words'] = df['text'].str.split().str.len()
ratings_sorted = sorted(df['rating'].unique())
groups   = [df[df['rating'] == r]['n_words'].tolist() for r in ratings_sorted]
colors_g = [NEG, NEG, MID, POS, POS]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bp = axes[0].boxplot(
    groups, tick_labels=[f'{r} estrellas' for r in ratings_sorted],
    patch_artist=True, medianprops=dict(color='black', linewidth=2)
)
for patch, color in zip(bp['boxes'], colors_g):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0].set_title('Amazon — longitud por rating')
axes[0].set_ylabel('palabras')

mu_amz = df['n_words'].mean()
axes[1].hist(df['n_words'].clip(upper=400), bins=40, color=BASE, edgecolor='white')
axes[1].axvline(mu_amz, color='crimson', linestyle='--', label=f'media={mu_amz:.1f}')
axes[1].set_title('Amazon — longitud (zoom <= 400 palabras)')
axes[1].set_xlabel('palabras')
axes[1].legend()

plt.tight_layout()
plt.show()
print(df.groupby('rating')['n_words'].agg(['mean', 'median', 'max']).round(1).to_string())

### 2.3 Vocabulario por sentimiento

A diferencia de Twitter, el vocabulario de Amazon es más formal y descriptivo. Los bigramas capturan frases de evaluación de producto (*highly recommend*, *waste of money*) que son señales muy fuertes y apenas aparecen si se miran solo unigramas.

In [ ]:
neg_amz = df[df['rating'] <= 2]['text'].tolist()
pos_amz = df[df['rating'] >= 4]['text'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hbar(axes[0], top_n(neg_amz), NEG, f'Amazon — top palabras (Negative ≤2★, n={len(neg_amz)})')
hbar(axes[1], top_n(pos_amz), POS, f'Amazon — top palabras (Positive ≥4★, n={len(pos_amz)})')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hbar(axes[0], top_bigrams(neg_amz), NEG, 'Amazon — top bigramas (Negative ≤2★)')
hbar(axes[1], top_bigrams(pos_amz), POS, 'Amazon — top bigramas (Positive ≥4★)')
plt.tight_layout()
plt.show()

### 2.4 Calidad del dataset

El corpus local proviene de un sampling de la categoría Books. Vale la pena verificar duplicados y textos anómalos antes de usarlo en producción.

In [ ]:
dupes_amz = len(df) - df['text'].nunique()
empty_amz = df['text'].str.strip().eq('').sum()
short_amz = int((df['n_words'] < 5).sum())

print('── Calidad Amazon ───────────────────────────────────')
print(f'  Total                : {len(df):,}')
print(f'  Duplicados           : {dupes_amz:,}')
print(f'  Vacíos               : {empty_amz:,}')
print(f'  Muy cortos (<5 pal.) : {short_amz:,}')
print()

df[['rating', 'title', 'text']].sample(5, random_state=42).assign(
    text=lambda d: d['text'].str[:100] + '...'
)

---
## 3. Resumen comparativo

Tabla con las métricas clave de ambos datasets para referencia al diseñar los pipelines de entrenamiento.

In [ ]:
tw_len    = [len(t.split()) for t in tr['text']]
tw_mean   = np.mean(tw_len)
tw_median = int(np.median(tw_len))
tw_p95    = int(np.percentile(tw_len, 95))
tw_total  = len(tr['text'])
tw_vocab  = len(set(w for t in tr['text'] for w in t.lower().split()))

amz_mean   = df['n_words'].mean()
amz_median = int(df['n_words'].median())
amz_p95    = int(df['n_words'].quantile(0.95))
amz_vocab  = len(set(w for t in df['text'] for w in t.lower().split()))

pct_mention = sum(has_mention) / len(tw_texts) * 100
pct_hashtag = sum(has_hashtag) / len(tw_texts) * 100
pct_url     = sum(has_url)     / len(tw_texts) * 100

summary = pd.DataFrame({
    'Metrica': [
        'Total ejemplos',
        'Clases',
        'Longitud media (palabras)',
        'Longitud mediana',
        'Percentil 95 de longitud',
        'Duplicados',
        'Vocabulario unico',
        '% con @menciones',
        '% con #hashtags',
        '% con URLs',
    ],
    'Twitter (train)': [
        f'{tw_total:,}',
        '2 (pos / neg)',
        f'{tw_mean:.1f}',
        f'{tw_median}',
        f'{tw_p95}',
        f'{dupes_tw:,}',
        f'{tw_vocab:,}',
        f'{pct_mention:.1f}%',
        f'{pct_hashtag:.1f}%',
        f'{pct_url:.1f}%',
    ],
    'Amazon': [
        f'{len(df):,}',
        '5 ratings (binarizable)',
        f'{amz_mean:.1f}',
        f'{amz_median}',
        f'{amz_p95}',
        f'{dupes_amz:,}',
        f'{amz_vocab:,}',
        '-',
        '-',
        '-',
    ],
})
summary.set_index('Metrica')